# Growing Neural Cellular Automata

Implementation of "Growing Neural Cellular Automata" (Mordvintsev et al., 2020) 
in PyTorch — a model that learns to grow, persist, and regenerate complex 
patterns from a single cell using only local update rules.

In [ ]:
import numpy as np
import torch
from PIL import Image
import io, os

from torch.nn import MSELoss
import matplotlib.pyplot as plt

# Set a random seed for reproducibility
np.random.seed(42)

# Set device to cuda if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# We are going to consider squared grid of dimension n x n
H = 64
W = 64
N_CHANNEL = 16

epochs = 8000
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

img_path = r'/content/sample_data/gengar_gen_aut.png'

seed = np.zeros([N_CHANNEL,H, W], np.float32)
seed[3:,H//2, W//2] = 1.0 # Neglect first three channels, since they are RGB and we want to set them to zero.



# We want to simulate growth from a single cell, thus no other one should partecipate or
# carry any hidden state. Enforce this by setting all empty cell channels to zero.
# A cell is defined as empty is there is no mature (alpha > 0.1) cell in the 3x3 neighbour


Using device: cuda


In [ ]:
# Utility functions 

def to_rgba(x):
    return x[:,:4,:,:]


def to_alpha(x):
    return torch.clamp(x[3:4,:,:], 0.0, 1.0)


def to_rgb(x):
    rgb, a = x[:3,:,:], to_alpha(x)
    return torch.clamp(1.0-a+rgb, 0.0, 1.0)


def extract_rgb_from_state(x):
    x = x.squeeze(dim=0)
    return x[:3,:,:]

def convert_to_numpy(x):
    x = x.transpose(0,1)
    x = x.transpose(1,2)
    x = torch.clamp(x, 0.0, 1.0)
    x = (x.detach().cpu().numpy()*255.0).astype(np.uint8)
    return x

def save_as_img(x, outpath):
    im = Image.fromarray(x)
    im.save(outpath)


def load_image_target(img_path : str):

    # Load image from path
    img = Image.open(img_path).convert("RGBA")

    # Resize image to fit the state grid
    img = img.resize(size=[H,W])

    # Normalize pixel values after resizing
    img = np.float32(img)/255.0

    # Convert to tensor and reshape to desired format
    img_tensor = torch.from_numpy(img)
    img_tensor = img_tensor.transpose(0,-1)
    img_tensor = img_tensor.transpose(1,2)
    # Premultiply by alpha

    img_tensor[:3,:,:] *= img_tensor[3:,:,:]
    img_tensor = img_tensor.unsqueeze(dim=0)
    return img_tensor



# 1. Growth

In [ ]:
# update rule and Cellular Automata network for experiment 1

def mask_is_alive(state_grid : torch.tensor) -> torch.tensor:
    alive = torch.max_pool2d(state_grid[:,3:4,:,:], kernel_size=(3,3), padding=1, stride = 1) > 0.1
    alive = alive.float()
    #alive = alive.unsqueeze(dim=1)

    state_grid = state_grid * alive
    return alive

def update(
        x: torch.tensor,
        dnn: torch.nn,
        fire_rate: float = 0.5,
        step_size: int = 0.1):
    # 1. Randomly sample the num of steps
    steps = np.random.choice(range(64,96+1))

    x = x.to(device)
    dnn = dnn.to(device)

    # 2. Get pre living mask
    pre_living_mask  = mask_is_alive(x)


    # 4. Run updates
    for step in range(steps):
        # 5. Perception vector calculation
        y = dnn.perception(x)

        y = y.to(device)

        # 6. Calculate pixel update
        dx = dnn.forward(y)

        # 6.1 Mutiply update per step_size to tackle exploding gradient
        dx = dx * step_size

        # 7. Living mask
        shape_x = x[:, :1, :, :].shape
        #print(f"shape_x: {shape_x}")
        #update_mask = torch.tensor((np.random.uniform(shape_x) <= 0.5))
        update_mask = torch.rand(shape_x).to(device) <= fire_rate

        #print(f"update mask shape: {update_mask.shape}")
        x = x + dx * update_mask.float()

        post_living_mask = mask_is_alive(x)

        # This is causing the issue, setting the alive mask to zero
        life_mask = pre_living_mask * post_living_mask


        pre_living_mask = post_living_mask

    return x * life_mask.float()



class CANetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.layer1 = torch.nn.Conv2d(48,128, kernel_size=1)
        self.act1 = torch.nn.ReLU()
        self.layer2 = torch.nn.Conv2d(128,16, kernel_size=1)

        torch.nn.init.zeros_(self.layer2.weight)
        torch.nn.init.zeros_(self.layer2.bias)



        sobel_x = np.array([[-1, 0, +1],
                            [-2, 0, +2],
                            [-1, 0, +1]], dtype = np.float32)

        sobel_x_tensor = torch.from_numpy(sobel_x)
        sobel_x_tensor = sobel_x_tensor.unsqueeze(dim=0)
        sobel_x_tensor = sobel_x_tensor.unsqueeze(dim=0)
        sobel_x_tensor = sobel_x_tensor.repeat((N_CHANNEL, 1, 1, 1))
        sobel_x_tensor = sobel_x_tensor.contiguous() # Added .contiguous()

        sobel_y_tensor = sobel_x_tensor.transpose(dim0=-2, dim1=-1)
        sobel_y_tensor = sobel_y_tensor.contiguous() # Added .contiguous()

        self.register_buffer('sobel_x', sobel_x_tensor)
        self.register_buffer('sobel_y', sobel_y_tensor)

    def forward(self,x):
        x = self.layer1(x)
        x = self.act1(x)
        return self.layer2(x)

    def perception(self,
                   x: torch.tensor,
                   )-> torch.tensor:

        # calculate gradients via convolution with sobel tensors

        grad_x = torch.nn.functional.conv2d(x,self.sobel_x,groups=N_CHANNEL, padding=1)
        grad_y = torch.nn.functional.conv2d(x,self.sobel_y,groups=N_CHANNEL, padding=1)

        perception_vector = torch.cat([x,grad_x,grad_y], dim=1)
        perception_vector = perception_vector.float()
        return perception_vector


def train(
        H : int,
        W : int,
        N_CHANNEL: int,
        target_path: str,
        model: CANetwork,
        epochs: int = 1000,

            ):

    # Initialize predictions list
    preds = []

    # Extract filename from filepath
    filename = os.path.basename(target_path).split('_')[0]
    print(f'filename:{filename}')

    loss_fn = torch.nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr= 2e-3)
    losses = []
    #optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

    # prepare seed
    seed = np.zeros([N_CHANNEL,H, W], np.float32)
    # Neglect first three channels, since they are RGB and we want to set them to zero.
    seed[3:,H//2, W//2] = 1.0

    x = torch.from_numpy(seed)
    x = x.unsqueeze(dim=0).float()
    x = x.to(device)



    # load target image
    y = load_image_target(target_path)
    y = y.to(device)
    os.makedirs('outputs',exist_ok=True)

    for epoch in range(epochs):

        x_epoch = x.clone()
        optimizer.zero_grad()
        y_hat = update(x_epoch,model)

        #preds.append(y_hat)

        loss = loss_fn(y_hat[:,:4,:,:],y)
        losses.append(loss.item())

        # Setup for visualization
        if epoch%250 == 0 :
            print(f"Epoch {epoch}, Loss: {loss.item()}")
            pred_img = extract_rgb_from_state(y_hat)
            y_hat_vis = pred_img.detach()
            pred_img_np = convert_to_numpy(y_hat_vis)
            epoch_filename = f"{filename}_{epoch:04d}.png"
            basedir_first_exp = r'outputs/single_batch'
            os.makedirs(basedir_first_exp, exist_ok = True)
            out_path = os.path.join(basedir_first_exp, epoch_filename)
            save_as_img(pred_img_np,out_path)

        loss.backward()
        optimizer.step()




    last_loss = loss
    trained_nn = model
    return  trained_nn, last_loss, losses

# Single batch training

In [ ]:


model = CANetwork()
# Note, the larger the grid, the higher the numer of epochs required for convergence.
# As a reference, 40x40 grid required approx 1200 epochs. With 64x64 >1500.

trained_model, last_loss, all_losses = train(H, W, N_CHANNEL, img_path, model, epochs=epochs)
# Plotting loss vs epoch

epochs_range = np.arange(epochs)

fig = plt.plot(epochs_range,all_losses)
plt.xlabel("epochs")
plt.ylabel("loss")
plt.title("Loss vs epochs")

filename:gengar
Epoch 0, Loss: 0.6355249285697937


KeyboardInterrupt: 

In [ ]:
torch.save(trained_model,'single_batch_model.pkl')

# 2. Persistence & 3. Regrowth

## Multibatch training

In [ ]:
model_batch = CANetwork().to(device)



def create_pool(N_CHANNEL, H, W, POOL_SIZE):
    """ Create pool of initial states
        to be used for sampling.
        For the initial step, concerning
        this function, they should all
        be single pixel states
    """
    seed = np.zeros([N_CHANNEL,H, W], np.float32)
    # Neglect first three channels, since they are RGB and we want to set them to zero.
    seed[3:,H//2, W//2] = 1.0

    # Replicate pool for desired size
    pool = np.stack((seed.copy(),)*POOL_SIZE, axis=0)

    return seed, pool



def sample_pool(pool, BATCH_SIZE):
    """Sample indices from pool and correspoding values
       to create a batch for training
    """

    pool_size = pool.shape[0]
    sampled_idx = np.random.choice(pool_size,BATCH_SIZE)
    #batch = np.array([pool[i] for i in sampled_idx])
    batch = torch.from_numpy(pool[sampled_idx]).to(device)
    #batch = [seed] + batch

    return sampled_idx, batch



def replace_in_pool(
        pool,
        idxs,
        outputs
        ):
    """
       Replace batch used for
       training with outputs from
       same batch
    """

    pool[idxs] = outputs

    return pool


def make_circle_maks(BATCH_SIZE, W, H):

    x = torch.linspace(-1.0,1.0,W)[None,None,:]
    y = torch.linspace(-1.0,1.0,H)[None,:,None]

    center = torch.rand(size=[2,BATCH_SIZE,1,1]) -0.5 # subtract to ensure range -.5, .5 as from paper
    r = 0.1 + (0.4-0.1)* torch.rand([BATCH_SIZE,1,1])
    x, y = (x-center[0])/r, (y-center[1])/r
    mask = torch.tensor(x*x + y*y < 1, dtype=torch.float32)
    return mask.unsqueeze(dim=1)


def apply_circle_mask(x: torch.tensor, mask: torch.tensor):
    return x * (1 - mask)

def pool_training(
        N_CHANNEL,
        H,
        W,
        POOL_SIZE,
        training_iterations,
        BATCH_SIZE,
        target_path,
        model_batch,
        REGROWTH: bool = True

):
    """
    Batch training version of training function, for
    second experiment implementation.

    Goal of the second experiment is to make the target
    an attractor, increasing pattern stability over time
    """
    # Extract filename from filepath
    filename = os.path.basename(target_path).split('_')[0]
    print(f'filename:{filename}')

    # 1. Create state pool
    seed, pool = create_pool(N_CHANNEL,H,W, POOL_SIZE)
    seed_tensor = torch.tensor(seed).to(device)

    loss_fn = torch.nn.MSELoss(reduction='mean')
    loss_worst_sample = torch.nn.MSELoss(reduction='none')
    optimizer = torch.optim.Adam(model_batch.parameters(), lr= 2e-3)
    losses = []

    # Initialize predictions list
    #preds = []

    # load target image
    y = load_image_target(target_path)
    os.makedirs(r'outputs/batch_experiment',exist_ok=True)
    y_expanded = y.expand(BATCH_SIZE, -1, -1, -1)

    y_expanded = y_expanded.to(device)


    for i in range(training_iterations):
        # 2. sample a batch from the pool
        idx, batch = sample_pool(pool, BATCH_SIZE)
        # Calcualte loss in batch

        tensor_batch = torch.tensor(batch).to(device)
        #batch_losses = np.array([loss_fn(tensor_batch[i].unsqueeze(dim=0)[:,:4,:,:], y) for i in range(tensor_batch.shape[0])])
        #sorted_losses = sorted([np.double(i) for i in batch_losses], reverse=True)
        batch_losses = loss_worst_sample(tensor_batch[:,:4,:,:], y_expanded)

        sample_losses = batch_losses.mean(dim=(1,2,3))

        # Determine min loss samples to damage
        if REGROWTH:
            # Create mask for damage
            _ , idx_ = torch.topk(sample_losses, k=4, largest=False)
            # Apply damage to batch images
            mask = make_circle_maks(BATCH_SIZE, W, H).to(device)
            print(mask.shape)
            tensor_batch[idx_] = apply_circle_mask(
                tensor_batch[idx_],
                mask[idx_]
            )

        # Replace worst performing index with seed
        max_loss_idx = sample_losses.argmax()
        tensor_batch[max_loss_idx] = seed_tensor

        x_epoch = tensor_batch.clone()
        x_epoch = x_epoch.to(device)

        optimizer.zero_grad()

        # Update must be modified since expectes (1,16,H,W) shape
        # Now we have a multibatch tensor (N_BATCH, 16, H, W)
        y_hat = update(x_epoch,model_batch)
        #preds.append(y_hat)

        # Setup for visualization
        loss = loss_fn(y_hat[:,:4,:,:], y_expanded).mean()

        if i%250 == 0 :
            print(f"Epoch {i}, Loss: {loss.item()}")
            for b in range(y_hat.shape[0]):
                pred_img = extract_rgb_from_state(y_hat[b].unsqueeze(0))
                y_hat_vis = pred_img.detach()
                pred_img_np = convert_to_numpy(y_hat_vis)
                epoch_filename = f"{filename}_{i:04d}.png"
                basedir = f'outputs\\batch_{b}'
                os.makedirs(basedir, exist_ok=True)
                out_path = os.path.join(basedir, epoch_filename)

                save_as_img(pred_img_np, out_path)



        losses.append(loss.item())
        loss.backward()
        optimizer.step()

        pool = replace_in_pool(pool, idx, y_hat.detach().cpu().numpy())


    trained_nn = model_batch
    if not REGROWTH:
        torch.save(trained_nn, 'multi_batch_model.pkl')
    else:
      torch.save(trained_nn, 'regrowth_model.pkl')
    return  trained_nn, losses

In [ ]:
trained_nn, losses = pool_training(N_CHANNEL,H,W,100,8500,8,target_path=img_path,model_batch=model_batch, REGROWTH=False)

filename:gengar
Epoch 0, Loss: 0.6355248689651489


/tmp/ipykernel_334/778797532.py:117: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tensor_batch = torch.tensor(batch).to(device)


Epoch 250, Loss: 0.10947959125041962
Epoch 500, Loss: 0.07512542605400085
Epoch 750, Loss: 0.06530661880970001
Epoch 1000, Loss: 0.05013793334364891
Epoch 1250, Loss: 0.05011870712041855
Epoch 1500, Loss: 0.041207633912563324
Epoch 1750, Loss: 0.03837529569864273
Epoch 2000, Loss: 0.037424154579639435
Epoch 2250, Loss: 0.04040830582380295
Epoch 2500, Loss: 0.036087505519390106
Epoch 2750, Loss: 0.032542210072278976
Epoch 3000, Loss: 0.031690813601017
Epoch 3250, Loss: 0.03109218366444111
Epoch 3500, Loss: 0.028552543371915817
Epoch 3750, Loss: 0.02941635623574257
Epoch 4000, Loss: 0.027584895491600037
Epoch 4250, Loss: 0.026074737310409546
Epoch 4500, Loss: 0.02757641300559044
Epoch 4750, Loss: 0.024829881265759468
Epoch 5000, Loss: 0.023663543164730072
Epoch 5250, Loss: 0.02146855555474758
Epoch 5500, Loss: 0.021517761051654816
Epoch 5750, Loss: 0.022007253021001816
Epoch 6000, Loss: 0.019833609461784363
Epoch 6250, Loss: 0.018426597118377686
Epoch 6500, Loss: 0.017376545816659927
Epo

### Test model persistency over multiple epochs post training

In [ ]:
@torch.no_grad()
def persistency_test(seed, model, n_steps, filename):
    x = torch.tensor(seed).unsqueeze(dim=0)
    for step in range(n_steps):
        y_hat = update(x,model)
        x = y_hat
        if step%5 == 0 :
            print(f"Update step:{step}")
            pred_img = extract_rgb_from_state(y_hat)
            y_hat_vis = pred_img.detach()
            pred_img_np = convert_to_numpy(y_hat_vis)
            step_filename = f"{filename}_{step:04d}.png"
            basedir = '/content/outputs/persistency_test'
            os.makedirs(basedir, exist_ok=True)
            out_path = os.path.join(basedir, step_filename)
            save_as_img(pred_img_np, out_path)

seed = np.zeros([N_CHANNEL,H, W], np.float32)
seed[3:,H//2, W//2] = 1.0

persistency_test(seed, trained_nn, 1000, "persistency_multi_batch" )
#persistency_test(seed, trained_model, 1000, "persistency_single_batch" )



Update step:0
Update step:50
Update step:100
Update step:150
Update step:200
Update step:250
Update step:300
Update step:350
Update step:400
Update step:450
Update step:500
Update step:550
Update step:600
Update step:650
Update step:700
Update step:750
Update step:800
Update step:850
Update step:900
Update step:950


### Test model regrowth capabilities after training, by applying circular damages and updating it

In [ ]:
@torch.no_grad()
def regrowth_test(seed, model, n_steps, damage_step, filename):
    x = torch.tensor(seed).unsqueeze(dim=0).to(device)
    for step in range(n_steps):
        x = update(x, model)

        if step == damage_step:
            mask = make_circle_maks(1, W, H).to(device)
            x = apply_circle_mask(x, mask)
            pred_img = extract_rgb_from_state(x)
            
            pred_img_np = convert_to_numpy(pred_img.detach())
            save_as_img(pred_img_np, os.path.join(basedir, f"{filename}_{step:04d}_damage.png"))

        if step % 5 == 0:
            print(f"Step: {step}")
            pred_img = extract_rgb_from_state(x)
            pred_img_np = convert_to_numpy(pred_img.detach())
            step_filename = f"{filename}_{step:04d}.png"
            basedir = 'outputs/regrowth_test'
            os.makedirs(basedir, exist_ok=True)
            out_path = os.path.join(basedir, step_filename)
            save_as_img(pred_img_np, out_path)


In [ ]:
regrowth_test(seed, trained_nn, 1000, 50, 'regrowth_multi_batch')

Step: 0
Step: 5
Step: 10
Step: 15
Step: 20
Step: 25
Step: 30
Step: 35
Step: 40
Step: 45


/tmp/ipykernel_334/778797532.py:62: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask = torch.tensor(x*x + y*y < 1, dtype=torch.float32)


Step: 50
Step: 55
Step: 60
Step: 65
Step: 70
Step: 75
Step: 80
Step: 85
Step: 90
Step: 95
Step: 100
Step: 105
Step: 110
Step: 115
Step: 120
Step: 125
Step: 130
Step: 135
Step: 140
Step: 145
Step: 150
Step: 155
Step: 160
Step: 165
Step: 170
Step: 175
Step: 180
Step: 185
Step: 190
Step: 195
Step: 200
Step: 205
Step: 210
Step: 215
Step: 220
Step: 225
Step: 230
Step: 235
Step: 240
Step: 245
Step: 250
Step: 255
Step: 260
Step: 265
Step: 270
Step: 275
Step: 280
Step: 285
Step: 290
Step: 295
Step: 300
Step: 305
Step: 310
Step: 315
Step: 320
Step: 325
Step: 330
Step: 335
Step: 340
Step: 345
Step: 350
Step: 355
Step: 360
Step: 365
Step: 370
Step: 375
Step: 380
Step: 385
Step: 390
Step: 395
Step: 400
Step: 405
Step: 410
Step: 415
Step: 420
Step: 425
Step: 430
Step: 435
Step: 440
Step: 445
Step: 450
Step: 455
Step: 460
Step: 465
Step: 470
Step: 475
Step: 480
Step: 485
Step: 490
Step: 495
Step: 500
Step: 505
Step: 510
Step: 515
Step: 520
Step: 525
Step: 530
Step: 535
Step: 540
Step: 545
Step: 550


In [1]:
import os, re

folder = "../output/single_batch"
files = sorted(
    [f for f in os.listdir(folder) if re.match(r"gengar_\d+\.png", f)],
    key=lambda f: int(re.match(r"gengar_(\d+)\.png", f).group(1))
)

for i, fname in enumerate(files):
    new_name = f"frame_{i:04d}.png"
    os.rename(os.path.join(folder, fname), os.path.join(folder, new_name))

In [5]:
folder = "../output/regrowth_test"
pattern = re.compile(r"regrowth_multi_batch_(\d+)(_damage)?\.png")

files = []
for f in os.listdir(folder):
    m = pattern.match(f)
    if m:
        num = int(m.group(1))
        is_damage = m.group(2) is not None
        # sort key: damage frame comes right after its matching number
        files.append((num, is_damage, f))

files.sort(key=lambda x: (x[0], x[1]))

for i, (num, is_damage, fname) in enumerate(files):
    new_name = f"frame_{i:04d}.png"
    os.rename(os.path.join(folder, fname), os.path.join(folder, new_name))